In [4]:
# cell: NB_seg (Colab GPU)
import os, sys
from pathlib import Path
ROOT = '/content/drive/MyDrive/NCKH_PanDerm'   # Drive đã mount bằng lệnh extension
CODE = '/content/nckh'                          # bản clone chỉ đọc; sửa code ở máy rồi push
assert os.path.isdir(ROOT), 'Chưa mount Drive: Ctrl+Shift+P → Colab: Mount Google Drive to Server...'
os.environ['NCKH_ROOT'] = ROOT
os.environ['NCKH_LOCAL_DATA'] = '/content/data'
!test -d {CODE}/.git || git clone -q https://github.com/NeuroDev204/nckh.git {CODE}
!git -C {CODE} pull -q --ff-only && git -C {CODE} log --oneline -1
!rm -f {CODE}/__init__.py   # file thừa ở gốc repo che mất src/nckh (main cũ); no-op khi main đã sửa
sys.path.insert(0, f'{CODE}/src')
for m in [m for m in sys.modules if m.split('.')[0] == 'nckh']: del sys.modules[m]   # bỏ nckh sai đã cache, khỏi restart

59ab5a8 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #2 from NeuroDev204/docs/sv-b-guides


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
CK = f'{ROOT}/checkpoints/panderm_bb_data6_checkpoint-499.pth'
!mkdir -p {ROOT}/checkpoints
!test -f {CK} || gdown 17J4MjsZu3gdBP6xAQi_NMDVvH65a00HB -O {CK}
!ls -la {CK}
from nckh.runcard import sha256_file
digest = sha256_file(Path(CK))
print(digest)
with open (f'{ROOT}/checkpoints/SHA256SUMS', 'a') as fh:
    fh.write(f'{digest} panderm_bb_data6_checkpoint-499.pth #tai {__import__("datetime").date.today()}\n')


-rw------- 1 root root 343298382 Apr 29  2025 /content/drive/MyDrive/NCKH_PanDerm/checkpoints/panderm_bb_data6_checkpoint-499.pth
be1e0fb108b3bc58721cb5195f136c948160799438f222acf1fd142230ac1ff1


In [6]:
# cell: NB_seg (Colab GPU)
VENV = '/content/venv_seg'
CK = f'{ROOT}/checkpoints/panderm_bb_data6_checkpoint-499.pth'
!test -d /content/PanDerm || git clone -q https://github.com/SiyuanYan1/PanDerm.git /content/PanDerm
!cd /content/PanDerm && git checkout -q fd7a80748ba7fc3e203fed88f909f4689d0d6f24 && git log --oneline -1
!pip install -q uv
!uv python install 3.10
!test -x {VENV}/bin/python || uv venv -q --python 3.10 --seed {VENV}
# setuptools<81: mmengine/mmseg còn dùng pkg_resources đã bị bỏ ở setuptools 81.
!uv pip install -q --python {VENV}/bin/python "setuptools<81" wheel
!uv pip install -q --python {VENV}/bin/python torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
!uv pip install -q --python {VENV}/bin/python mmengine==0.10.4 mmcv==2.1.0 mmsegmentation==1.2.2 --find-links https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
!uv pip install -q --python {VENV}/bin/python -r /content/PanDerm/segmentation/requirements.txt openpyxl
!uv pip install -q --python {VENV}/bin/python -e {CODE}
!{VENV}/bin/python -c "import torch, mmcv, mmseg; print('torch', torch.__version__, '| mmcv', mmcv.__version__, '| mmseg', mmseg.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')"

fd7a807 (HEAD, origin/main, origin/HEAD, main) Enhance formatting of updates in README.md
Python 3.10 is already installed
torch 2.1.2+cu118 | mmcv 2.1.0 | mmseg 1.2.2 | GPU Tesla T4


In [10]:
!cd {CODE} && git pull
!{VENV}/bin/python {CODE}/scripts/inspect_checkpoint.py {CK}

remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 12 (delta 5), reused 11 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 4.26 KiB | 1.06 MiB/s, done.
From https://github.com/NeuroDev204/nckh
   59ab5a8..e78efd1  main             -> origin/main
   62f4b69..631589c  docs/sv-b-guides -> origin/docs/sv-b-guides
Updating 59ab5a8..e78efd1
Fast-forward
 .../2026-10-07-sv-b-local-vscode-colab-plan.md     | 764 +++++++++++++++++++++
 .../2026-10-07-sv-b-local-vscode-colab-design.md   |  75 ++
 docs/sv_b/00_tong_quan_va_lo_trinh.md              | 218 ++++--
 docs/sv_b/P0_P1_P4_P11_vai_tro_ho_tro.md           |  28 +-
 docs/sv_b/P10_tai_lap_hinh_bang.md                 |  57 +-
 docs/sv_b/P2_moi_truong_checkpoint_smoke_test.md   | 225 +++---
 docs/sv_b/P3_du_lieu_manifest_split.md             | 105 ++-
 docs/sv_b/P5a_segmentation_isic2018.md             |  75 +-
 docs/sv_b/P5b_class